In [ ]:
!pip install optuna -q

In [ ]:
import optuna
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

In [ ]:
def objective_cat(trial):

    params = {
        "iterations": trial.suggest_int(
            "iterations",
            1000,
            6000
        ),
        "depth": trial.suggest_int(
            "depth",
            4,
            12
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.1,
            log=True
        ),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            1,
            20
        ),
        "bagging_temperature": trial.suggest_float(
            "bagging_temperature",
            0,
            10
        ),
        "loss_function": "RMSE",
        "task_type": "GPU",
        "devices": "0",
        "verbose": 0
    }

    kf = KFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    scores = []

    for tr_idx,val_idx in kf.split(X):

        X_train = X.iloc[tr_idx]
        X_valid = X.iloc[val_idx]

        y_train = y.iloc[tr_idx]
        y_valid = y.iloc[val_idx]

        model = CatBoostRegressor(
            **params
        )

        model.fit(
            X_train,
            y_train
        )

        pred = model.predict(
            X_valid
        )

        score = r2_score(
            y_valid,
            pred
        )

        scores.append(score)

    return np.mean(scores)

study_cat = optuna.create_study(
    direction="maximize"
)

study_cat.optimize(
    objective_cat,
    n_trials=30
)

print(study_cat.best_params)

In [ ]:
best_cat = CatBoostRegressor(
    **study_cat.best_params,
    loss_function="RMSE",
    task_type="GPU",
    devices="0",
    verbose=0
)

best_cat.fit(
    X,
    y
)

pred_cat = best_cat.predict(
    X_test
)

In [ ]:
def objective_xgb(trial):

    params = {

        "n_estimators": trial.suggest_int(
            "n_estimators",
            500,
            5000
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            12
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.1,
            log=True
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        "tree_method":"hist",
        "device":"cuda",
        "random_state":42
    }

    kf = KFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    scores = []

    for tr_idx,val_idx in kf.split(X):

        model = XGBRegressor(
            **params
        )

        model.fit(
            X.iloc[tr_idx],
            y.iloc[tr_idx]
        )

        pred = model.predict(
            X.iloc[val_idx]
        )

        score = r2_score(
            y.iloc[val_idx],
            pred
        )

        scores.append(score)

    return np.mean(scores)

study_xgb = optuna.create_study(
    direction="maximize"
)

study_xgb.optimize(
    objective_xgb,
    n_trials=20
)

print(study_xgb.best_params)

In [ ]:
best_xgb = XGBRegressor(
    **study_xgb.best_params,
    tree_method="hist",
    device="cuda",
    random_state=42
)

best_xgb.fit(
    X,
    y
)

pred_xgb = best_xgb.predict(
    X_test
)

In [ ]:
def objective_lgb(trial):

    params = {

        "n_estimators": trial.suggest_int(
            "n_estimators",
            500,
            5000
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            12
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.1,
            log=True
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves",
            16,
            256
        )
    }

    kf = KFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    scores = []

    for tr_idx,val_idx in kf.split(X):

        model = LGBMRegressor(
            **params
        )

        model.fit(
            X.iloc[tr_idx],
            y.iloc[tr_idx]
        )

        pred = model.predict(
            X.iloc[val_idx]
        )

        score = r2_score(
            y.iloc[val_idx],
            pred
        )

        scores.append(score)

    return np.mean(scores)

study_lgb = optuna.create_study(
    direction="maximize"
)

study_lgb.optimize(
    objective_lgb,
    n_trials=20
)

print(study_lgb.best_params)

In [ ]:
best_lgb = LGBMRegressor(
    **study_lgb.best_params
)

best_lgb.fit(
    X,
    y
)

pred_lgb = best_lgb.predict(
    X_test
)

In [ ]:
final_predictions = (
    0.45 * pred_cat
    + 0.35 * pred_xgb
    + 0.20 * pred_lgb
)

In [ ]:
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": final_predictions
})

submission.to_csv(
    "submission_optuna.csv",
    index=False
)

print(submission.shape)
print(submission.head())